# 09 RAG pipeline
Parse, chunk, embed, index, retrieve and answer with citations. Then break a stage on purpose.

Step 1: set up the kit

In [1]:
# Setup: in Colab, clone the kit first. Locally, find the kit folder. Change REPO_URL to your fork.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/arpitRanjan/atp-builder-kit"
if "google.colab" in sys.modules and not Path("kit/llm.py").exists():
    if not Path("atp-builder-kit").exists():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("atp-builder-kit")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-genai==3.3.0", "openai==3.51.0"], check=True)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "kit" / "llm.py").exists())
sys.path.insert(0, str(ROOT))
CORPUS = Path(os.environ.get("CORPUS_DIR", ROOT / "corpus"))
OUTPUT = Path(os.environ.get("KIT_OUTPUT_DIR", ROOT / "stage-09"))
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Kit folder found:", ROOT.name)

Kit folder found: atp-builder-kit


Step 2: parse the pages and their front matter

In [2]:
import json, re, time
import numpy as np
import pandas as pd
from kit.llm import call_llm
from kit.prices import cost
from kit.rag import get_embedder

def parse(folder):
    pages = []
    for path in sorted(Path(folder).glob("*.md")):
        if path.name.lower() == "readme.md":
            continue
        _, head, body = path.read_text().split("---", 2)
        meta = {k.strip(): v.strip() for k, v in (line.split(":", 1) for line in head.strip().splitlines())}
        pages.append({"id": meta["id"], "title": meta["title"], "visibility": meta["visibility"],
                      "body": " ".join(body.split())})
    return pages

pages = parse(CORPUS)
questions = [json.loads(line) for line in open(CORPUS / "questions.jsonl")]
print(len(pages), "pages,", len(questions), "labeled questions")
pd.DataFrame(pages)[["id", "title", "visibility"]].head()

15 pages, 25 labeled questions


,id,title,visibility
0,qn-01,Create and edit notes,public
1,qn-02,Export a note as PDF,public
2,qn-03,Tags and folders,public
3,qn-04,Search your notes,public
4,qn-05,Fix sync problems,public


Step 3: chunk: size and overlap are parameters

In [3]:
SIZE, OVERLAP = 120, 30  # in words

def chunk(pages, size=SIZE, overlap=OVERLAP):
    chunks = []
    for page in pages:
        words = page["body"].split()
        for n, start in enumerate(range(0, max(len(words) - overlap, 1), max(1, size - overlap))):
            chunks.append({"id": f"{page['id']}#{n}", "page_id": page["id"], "title": page["title"],
                           "text": " ".join(words[start:start + size])})
    return chunks

chunks = chunk(pages)
print(len(chunks), "chunks, for example", chunks[0]["id"], ":", chunks[0]["text"][:120])

30 chunks, for example qn-01#0 : To create a note: 1. Open Quillnote and sign in. 2. Select **New Note**. 3. Type a title at the top. 4. Add your text in


Step 4: embed and store in a local numpy index

In [4]:
embedder = get_embedder()  # all-MiniLM-L6-v2, or a hashing stand-in when KIT_OFFLINE=1
print("Embedder:", embedder.name)

def build_index(chunks):
    return {"chunks": chunks, "vectors": embedder.encode([c["text"] for c in chunks])}

def retrieve(index, query, k=3):
    scores = index["vectors"] @ embedder.encode([query])[0]  # vectors are normalized, so this is cosine
    return [index["chunks"][i] | {"score": round(float(scores[i]), 3)} for i in np.argsort(-scores)[:k]]

index = build_index(chunks)  # kept in memory: it rebuilds in seconds, so there is no binary file to commit
print("Index shape:", index["vectors"].shape)
[(h["id"], h["score"]) for h in retrieve(index, "How do I export a note as PDF?")]

Embedder: hash-384
Index shape: (30, 384)


[('qn-02#0', 0.4), ('qn-02#1', 0.331), ('qn-01#0', 0.223)]

Step 5: answer with citations to chunk ids

In [5]:
SYSTEM = ("Answer using only the numbered sources. Cite the source ids you used in square brackets, "
          "like [qn-02#0]. If the sources do not answer the question, say so.")

def answer(question, sources):
    lines = "\n".join(f"[{s['id']}] {s['text']}" for s in sources)
    r = call_llm(f"Sources:\n{lines}\n\nQuestion: {question}", system=SYSTEM, max_output_tokens=1000)
    r["cited_chunks"] = [c for c in dict.fromkeys(re.findall(r"qn-\d+#\d+", r["text"])) if c in {s["id"] for s in sources}]
    cited_pages = {c.split("#")[0] for c in re.findall(r"qn-\d+(?:#\d+)?", r["text"])}
    return r, cited_pages

r, cited = answer(questions[2]["question"], retrieve(index, questions[2]["question"]))
print(questions[2]["question"], "\n", r["text"], "\nCited chunks:", r["cited_chunks"], "| pages:", cited)

How do I export one of my notes as a PDF file? 
 To export a note as a PDF in Quillnote: 1. Open the note you want to save. 2. Check the note layout and make any final edits. 3. Select **More** [qn-02#0] 
Cited chunks: ['qn-02#0'] | pages: {'qn-02'}


Step 6: run every labeled question through the pipeline

In [6]:
def run_rag(index, label):
    rows = []
    for q in questions:
        hits = retrieve(index, q["question"])
        r, cited = answer(q["question"], hits)
        rows.append({"qid": q["qid"], "setup": label, "correct": bool(cited & set(q["page_ids"])),
                     "retrieved": [h["id"] for h in hits], "cited_chunks": r["cited_chunks"], "cited": sorted(cited),
                     "answer": r["text"],
                     "input_tokens": r["input_tokens"], "cost_usd": cost(r["input_tokens"], r["output_tokens"], r["model"]),
                     "latency_s": r["latency_s"]})
    return pd.DataFrame(rows)

good = run_rag(index, f"RAG, {SIZE} word chunks")
print("Accuracy:", round(good.correct.mean(), 2))
# The saved run: every answer with the chunk ids it cites.
good[["qid", "retrieved", "cited_chunks", "correct", "answer"]].to_json(OUTPUT / "rag_answers.jsonl", orient="records", lines=True)
print("Answers with cited chunk ids saved to", OUTPUT.name + "/rag_answers.jsonl")

Accuracy: 0.84
Answers with cited chunk ids saved to stage-09/rag_answers.jsonl


Step 7: break a stage: a bad chunk size

In [7]:
tiny = build_index(chunk(pages, size=8, overlap=0))
bad_size = run_rag(tiny, "RAG, 8 word chunks")
print("Accuracy with 8 word chunks:", round(bad_size.correct.mean(), 2), "vs", round(good.correct.mean(), 2))
broke = good.merge(bad_size, on="qid", suffixes=("_good", "_bad"))
broke = broke[broke.correct_good & ~broke.correct_bad]
if len(broke):
    row = broke.iloc[0]
    q = next(x for x in questions if x["qid"] == row.qid)
    print(f"\n{q['qid']}: {q['question']} (answer is on {q['page_ids']})")
    print("Wrong answer:", row.answer_bad)
    print("Traced to retrieval, these 8 word chunks came back:")
    for h in retrieve(tiny, q["question"]):
        print("  ", h["id"], "|", h["text"])

Accuracy with 8 word chunks: 0.6 vs 0.84

q01: How do I create a new note and rename it later? (answer is on ['qn-01'])
Wrong answer: - Create a new note. - Choose **From [qn-12#16]
Traced to retrieval, these 8 word chunks came back:
   qn-12#16 | - Create a new note. - Choose **From
   qn-08#8 | usual. - Create new notes, checklists, and drafts.
   qn-01#0 | To create a note: 1. Open Quillnote and


Step 8: break a stage: a missing document

In [8]:
MISSING = "qn-07"  # say the sync job skipped this page
partial = build_index(chunk([p for p in pages if p["id"] != MISSING]))
missing = run_rag(partial, f"RAG, {MISSING} missing")
needs = [x["qid"] for x in questions if MISSING in x["page_ids"]]
print("Accuracy on the questions that need", MISSING, ":", round(good[good.qid.isin(needs)].correct.mean(), 2),
      "before,", round(missing[missing.qid.isin(needs)].correct.mean(), 2), "after")
print("Accuracy on all questions:", round(missing.correct.mean(), 2), "(an average can hide a missing page)")
q = next(x for x in questions if x["page_ids"] == [MISSING])
row = missing[missing.qid == q["qid"]].iloc[0]
print(f"\n{q['qid']}: {q['question']}")
print("Answer:", row.answer)
print("Retrieved:", row.retrieved, "| page", MISSING, "in the index:",
      any(c["page_id"] == MISSING for c in partial["chunks"]))

Accuracy on the questions that need qn-07 : 0.5 before, 0.5 after
Accuracy on all questions: 0.88 (an average can hide a missing page)

q11: I deleted a note by accident. Can I get it back?
Answer: - Wait a moment for sync to start. - Let the app finish uploading your offline changes. Tips: - Keep the app updated so offline storage works smoothly. - Open [qn-08#1]
Retrieved: ['qn-01#1', 'qn-08#1', 'qn-08#0'] | page qn-07 in the index: False


Step 9: the same questions with the whole corpus in the prompt

In [9]:
whole = [{"id": p["id"], "text": p["title"] + ". " + p["body"]} for p in pages]
rows = []
for q in questions:
    r, cited = answer(q["question"], whole)
    rows.append({"qid": q["qid"], "setup": "whole corpus in prompt", "correct": bool(cited & set(q["page_ids"])),
                 "input_tokens": r["input_tokens"], "cost_usd": cost(r["input_tokens"], r["output_tokens"], r["model"]),
                 "latency_s": r["latency_s"]})
full = pd.DataFrame(rows)

Step 10: compare accuracy and cost per question

In [10]:
compare = pd.concat([good, bad_size, missing, full]).groupby("setup", sort=False).agg(
    accuracy=("correct", "mean"), input_tokens=("input_tokens", "mean"),
    cost_per_question_usd=("cost_usd", "mean"), median_latency_s=("latency_s", "median"))
compare.round(6).to_csv(OUTPUT / "baseline_vs_retrieval.csv")  # whole corpus baseline vs the retrieval path
compare.round(6)

,accuracy,input_tokens,cost_per_question_usd,median_latency_s
setup,,,,
"RAG, 120 word chunks",0.84,515.52,0.000685,0.700
"RAG, 8 word chunks",0.60,98.44,0.000149,0.563
"RAG, qn-07 missing",0.88,515.40,0.000690,0.616
whole corpus in prompt,0.80,3990.24,0.004158,0.581
